# Step 1 — Data preprocessing and exploratory analysis

Turns the raw JDsearch text files into typed tables, removes the label leak in the raw slate order, splits users, and runs the exploratory analysis that drives every later design decision (in particular the history retention windows raised in the interim feedback).

### Setup (run first)

**On Google Colab:** *Runtime → Change runtime type → GPU* (T4 is enough; A100/L4 is faster). Put the two raw JDsearch
files in `MyDrive/bt4222/raw/` (`user_behavior_data.txt`, `product_meta_data.txt`). The cell mounts Drive, clones the
repository, installs the requirements and writes every output to `MyDrive/bt4222/outputs/`, so results persist between
sessions and the notebooks can be run one after another (or on different days).

**Locally:** run the notebook from the repository with the virtual environment active; data is read from `data/raw/`
and outputs go to `outputs/`.

`CONFIG = "configs/smoke.yaml"` runs everything on a 2,000-user sample with tiny models (minutes, CPU) to check the code;
`configs/default.yaml` is the real run. Every stage skips work whose outputs already exist; pass `"--force"` to redo one.

In [ ]:
import os, subprocess, sys
from pathlib import Path

CONFIG = "configs/default.yaml"          # "configs/smoke.yaml" for a quick check
REPO_URL = "https://github.com/clarequek/bt4222-two-tower-recsys.git"
DRIVE_DIR = "/content/drive/MyDrive/bt4222"   # Colab: raw/ (input) and outputs/ (written by the pipeline)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/bt4222-two-tower-recsys")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from jds.notebook import Runner, banner
run = Runner(CONFIG, drive_dir=DRIVE_DIR if IN_COLAB else None)

## 1.1 Preprocessing (`scripts/00_prepare.py`, `src/jds/prepare.py`)

What it does and why:
* **Parses** the user file (one row = one search session: test query, displayed slate with labels, history) and the
  4.8 GB product file (streamed, kept only for items that appear in the logs).
* **History timestamps.** `history_time_list` holds *gaps in seconds* between consecutive events, the last one being the
  gap to the test query. Summing from the end gives each event's **age in days before the test query** (`age_days`),
  the quantity the interim feedback asked us to trace.
* **Leakage control.** In the raw file every slate is sorted by label (positives first in 100% of slates), so the
  original position would give the answer away. Every slate is shuffled with a fixed seed; the original position is
  kept only for auditing and is never a feature.
* **Split by user** 80 / 10 / 10 into train / validation / test (only users with at least one positive are evaluated);
  training users get one of 2 folds for cross-fitting (out-of-fold CTR features and tower scores, see steps 2 and 5).

In [ ]:
run.stage("00_prepare.py")

In [ ]:
run.json("prepare_stats.json")

## 1.2 Exploratory analysis (`scripts/01_eda.py`, `src/jds/eda.py`)

**The history is three logs with different retention windows.** Counted back from the test query, clicks are only kept
for 7 days, carts and follows for 90 days, orders for 365 days. That is why orders and carts outnumber clicks overall:
beyond 7 days a missing click means *not recorded*, not *did not click*. Inside the 7-day window the funnel looks like a
normal e-commerce funnel.

In [ ]:
run.stage("01_eda.py")

In [ ]:
run.figure("history_timeline.png")
import pandas as pd
pd.DataFrame(run.json("eda.json")["retention"])

### Funnel by elapsed days (interim feedback)

Clicks per cart and carts per order in each age bucket. Shaded bands are typical e-commerce ranges (clicks:carts 5–10,
carts:orders 3–5). Read where the ratios fall off a cliff: that is where a type's log ends.

In [ ]:
run.figure("funnel_by_age.png")
run.table("eda_funnel_by_age.csv", highlight=())

In [ ]:
eda = run.json("eda.json")
print("History that can never have a recorded click:")
eda["missing_clicks"]

In [ ]:
print("Naive click ratio vs share of recent history (correlation):")
eda["recency_confound"]

### The supervised signal: displayed slates and graded labels

Labels: 0 ignored, 1 click, 2 cart, 3 order. How many positives a slate has, what the deepest engagement is, and how
many displayed items are *cold* (never in anyone's history) — the reason the item tower is built from content rather
than item-id embeddings.

In [ ]:
run.figure("slates_and_history.png")
eda["slates"]

In [ ]:
print("Item popularity as a ranking signal (pooled AUC), counted inside each type window vs all history:")
eda["popularity_auc"]